In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) return false;

In [ ]:
import dataset, plots
import pandas as pd
from pathlib import Path

from config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR

sp500_components = dataset.SP500.load_historical()

## Połączenie danych z Yahoo Finance i EODHD

Dane z obu źródeł są przechowywane w formacie kolumnowym — każdy plik CSV odpowiada jednej zmiennej (np. `Close.csv`, `Adj_Close.csv`), a kolumny oznaczają tickery. Poniżej definiujemy dwie funkcje:

- **`merge_price_data`** — łączy dwa słowniki ramek danych (Yahoo Finance i EODHD) w jeden słownik. Dla tickerów obecnych w obu źródłach preferowane są dane z EODHD.
- **`save_merged_data`** — zapisuje scalony słownik ramek do katalogu w tym samym formacie co pliki źródłowe w `data/external/`.

In [ ]:
def merge_price_data(
    df_yahoo: dict[str, pd.DataFrame],
    df_eodhd: dict[str, pd.DataFrame],
) -> dict[str, pd.DataFrame]:
    """Merge Yahoo Finance and EODHD price data into a single dict of DataFrames.

    For overlapping tickers, EODHD data takes precedence over Yahoo Finance.
    Columns present only in Yahoo Finance are appended as-is.

    Parameters
    ----------
    df_yahoo:
        Dict mapping column names (e.g. 'Close', 'Adj_Close') to wide DataFrames
        with dates as index and tickers as columns, as returned by
        ``dataset.YahooFinance.load()``.
    df_eodhd:
        Dict mapping column names to wide DataFrames, as returned by
        ``dataset.EODHD.load()``.

    Returns
    -------
    dict[str, pd.DataFrame]
        Merged dict in the same columnar format. For every column key that
        exists in at least one source, the result contains EODHD values where
        available and Yahoo Finance values elsewhere.
    """
    all_keys = set(df_yahoo) | set(df_eodhd)
    result: dict[str, pd.DataFrame] = {}

    for key in sorted(all_keys):
        yahoo_frame = df_yahoo.get(key, pd.DataFrame())
        eodhd_frame = df_eodhd.get(key, pd.DataFrame())

        if yahoo_frame.empty and eodhd_frame.empty:
            result[key] = pd.DataFrame()
            continue

        if yahoo_frame.empty:
            result[key] = eodhd_frame.copy()
            continue

        if eodhd_frame.empty:
            result[key] = yahoo_frame.copy()
            continue

        # Identify tickers present in both sources — EODHD takes priority
        overlapping = yahoo_frame.columns.intersection(eodhd_frame.columns)
        yahoo_only = yahoo_frame.columns.difference(eodhd_frame.columns)

        # Start from EODHD data; fill gaps with Yahoo Finance for EODHD tickers,
        # then append tickers that are only available from Yahoo Finance.
        eodhd_preferred = eodhd_frame.copy()
        if len(overlapping):
            eodhd_preferred[overlapping] = eodhd_frame[overlapping].combine_first(
                yahoo_frame[overlapping]
            )

        if len(yahoo_only):
            merged = pd.concat(
                [eodhd_preferred, yahoo_frame[yahoo_only]], axis=1
            ).sort_index()
        else:
            merged = eodhd_preferred.sort_index()

        result[key] = merged

    return result

In [ ]:
def save_merged_data(
    df: dict[str, pd.DataFrame],
    output_path: Path,
) -> None:
    """Save merged price data to CSV files in the same format as source files.

    Each key in the dict is saved as ``<key>.csv`` inside *output_path*,
    matching the columnar format used in ``data/external/yfinance/`` and
    ``data/external/eodhd/``.

    Parameters
    ----------
    df:
        Dict mapping column names to wide DataFrames (dates × tickers).
    output_path:
        Directory where the CSV files will be written. Created if it does
        not exist.
    """
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    for col, frame in df.items():
        if frame.empty or frame.dropna(how="all").empty:
            continue
        path = output_path / f"{col}.csv"
        frame.to_csv(path)
        print(f"Saved {col}.csv ({len(frame)} rows x {len(frame.columns)} columns)")

### Wczytanie danych źródłowych

In [ ]:
yf_data: dict[str, pd.DataFrame] = dataset.YahooFinance.load()
for column_name, frame in yf_data.items():
    if frame.empty:
        print(f"{column_name}: empty")
    else:
        print(f"{column_name}: {frame.shape}, {frame.index.min().date()} -> {frame.index.max().date()}")

In [ ]:
eodhd_data: dict[str, pd.DataFrame] = dataset.EODHD.load()
for column_name, frame in eodhd_data.items():
    if frame.empty:
        print(f"{column_name}: empty")
    else:
        print(f"{column_name}: {frame.shape}, {frame.index.min().date()} -> {frame.index.max().date()}")

### Scalanie i zapis

In [ ]:
merged_data = merge_price_data(yf_data, eodhd_data)

for column_name, frame in merged_data.items():
    if frame.empty:
        print(f"{column_name}: empty")
    else:
        print(f"{column_name}: {frame.shape}, {frame.index.min().date()} -> {frame.index.max().date()}")

In [ ]:
output_path = INTERIM_DATA_DIR / "merged"
save_merged_data(merged_data, output_path)

### Pokrycie danych

In [ ]:
coverage_df = plots.coverage_over_time(merged_data, sp500_components)
plots.summarize_df(coverage_df)

In [ ]:
plots.plot_missing_data_reasons("Merged")